# Breast Tumour Malignancy Screening — Model Development
**ML Assignment 2 · M.Tech (AIML/DSE) · BITS Pilani WILP**

Exploratory walkthrough of the five classifiers that back the Streamlit app.
The production training run lives in `model/train_models.py`; this notebook
reproduces the same pipeline step by step with the diagnostics attached.

**Positive class = Malignant (1).** All precision / recall / MCC figures below
refer to correctly identifying malignant tumours.

## 1 · Setup

In [ ]:
import json, warnings
from pathlib import Path

import joblib, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")

SEED, HOLDOUT, FOLDS = 17, 0.25, 5
TARGET = "diagnosis_malignant"
np.random.seed(SEED)
print("ready")

## 2 · Load and reframe the data

scikit-learn ships WDBC with `0 = malignant`, `1 = benign`. We invert it so the
clinically interesting event is the positive class.

In [ ]:
from sklearn.datasets import load_breast_cancer

bunch = load_breast_cancer(as_frame=True)
frame = bunch.frame.copy()
frame[TARGET] = (frame.pop("target") == 0).astype(int)
frame.columns = [c.strip().replace(" ", "_") for c in frame.columns]

print(f"shape            : {frame.shape}")
print(f"features         : {frame.shape[1]-1}   (assignment minimum: 12)")
print(f"instances        : {frame.shape[0]}   (assignment minimum: 500)")
print(f"missing values   : {frame.isna().sum().sum()}")
print(f"duplicate rows   : {frame.duplicated().sum()}")
frame[TARGET].value_counts().rename({0:"Benign",1:"Malignant"})

In [ ]:
frame.head()

In [ ]:
frame.describe().T.head(12)

### 2.1 Class balance and feature separability

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))

frame[TARGET].map({0:"Benign",1:"Malignant"}).value_counts().plot(
    kind="bar", ax=axes[0], color=["#4c72b0", "#c44e52"], rot=0)
axes[0].set_title("Class balance (37.3% malignant)")

sns.kdeplot(data=frame, x="worst_concave_points", hue=TARGET,
            fill=True, common_norm=False, ax=axes[1])
axes[1].set_title("worst_concave_points separates the classes cleanly")

sns.scatterplot(data=frame, x="mean_radius", y="mean_texture",
                hue=TARGET, alpha=.75, ax=axes[2])
axes[2].set_title("mean_radius vs mean_texture")
plt.tight_layout(); plt.show()

### 2.2 Multicollinearity check

The 30 columns are 10 base measurements × 3 summaries (mean / standard error /
worst). That design makes several columns near-duplicates — `mean_radius`,
`mean_perimeter` and `mean_area` are geometrically the same quantity. This is
worth noting up front: it is exactly the condition that hurts Gaussian Naive
Bayes (its independence assumption is badly violated) and it is why the
L1-penalised logistic model, which can zero out redundant columns, does so well.

In [ ]:
corr = frame.drop(columns=[TARGET]).corr()
plt.figure(figsize=(11, 8.5))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True,
            xticklabels=True, yticklabels=True,
            cbar_kws={"shrink": .6})
plt.xticks(fontsize=6); plt.yticks(fontsize=6)
plt.title("Feature correlation — three heavily redundant blocks")
plt.tight_layout(); plt.show()

high = (corr.abs().where(np.triu(np.ones(corr.shape), 1).astype(bool))
            .stack().sort_values(ascending=False))
print("Most collinear pairs:")
print(high.head(8).round(3))

## 3 · Stratified train / screening split

25% held out, stratified so both halves carry the same 37% malignant rate.
This held-out set is written to `test_data.csv` and is the file uploaded to
the Streamlit app — the app never sees the training rows.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

X = frame.drop(columns=[TARGET])
y = frame[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=HOLDOUT, stratify=y, random_state=SEED)

print(f"train    : {X_train.shape[0]:>3} patients, {int(y_train.sum())} malignant "
      f"({y_train.mean():.1%})")
print(f"screening: {X_test.shape[0]:>3} patients, {int(y_test.sum())} malignant "
      f"({y_test.mean():.1%})")

## 4 · Five classifiers, each tuned by 5-fold CV on ROC-AUC

Scaling is applied **only** to logistic regression and kNN. Trees and Gaussian
NB are scale-invariant, so a scaler there would be cargo-cult preprocessing.

Gaussian NB rather than Multinomial: WDBC features are continuous real-valued
measurements, and Multinomial NB assumes non-negative counts.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

SPECS = {
    "Logistic Regression": (
        Pipeline([("scale", StandardScaler()),
                  ("clf", LogisticRegression(max_iter=5000, solver="liblinear",
                                             random_state=SEED))]),
        {"clf__C": [0.05, 0.1, 0.5, 1.0, 5.0], "clf__penalty": ["l1", "l2"]}),
    "Decision Tree": (
        Pipeline([("clf", DecisionTreeClassifier(random_state=SEED))]),
        {"clf__criterion": ["gini", "entropy"],
         "clf__max_depth": [3, 4, 5, 7, None],
         "clf__min_samples_leaf": [1, 3, 5, 8]}),
    "kNN": (
        Pipeline([("scale", StandardScaler()), ("clf", KNeighborsClassifier())]),
        {"clf__n_neighbors": [3, 5, 7, 9, 11, 15],
         "clf__weights": ["uniform", "distance"], "clf__p": [1, 2]}),
    "Naive Bayes": (
        Pipeline([("clf", GaussianNB())]),
        {"clf__var_smoothing": np.logspace(-11, -6, 6)}),
    "Random Forest (Ensemble)": (
        Pipeline([("clf", RandomForestClassifier(random_state=SEED, n_jobs=-1))]),
        {"clf__n_estimators": [200, 400], "clf__max_depth": [None, 6, 10],
         "clf__min_samples_leaf": [1, 2, 4], "clf__max_features": ["sqrt", 0.4]}),
}

cv = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=SEED)
fitted, best_params = {}, {}

for name, (pipe, grid) in SPECS.items():
    search = GridSearchCV(pipe, grid, scoring="roc_auc", cv=cv, n_jobs=-1)
    search.fit(X_train, y_train)
    fitted[name] = search.best_estimator_
    best_params[name] = {k.replace("clf__", ""): v
                         for k, v in search.best_params_.items()}
    print(f"{name:<26} CV AUC={search.best_score_:.4f}  {best_params[name]}")

## 5 · Evaluation on the held-out screening set

In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef,
                             confusion_matrix, classification_report, roc_curve)

METRICS = ["Accuracy", "AUC", "Precision", "Recall", "F1", "MCC"]

rows = []
for name, model in fitted.items():
    pred  = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    rows.append({
        "ML Model Name": name,
        "Accuracy":  accuracy_score(y_test, pred),
        "AUC":       roc_auc_score(y_test, proba),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall":    recall_score(y_test, pred, zero_division=0),
        "F1":        f1_score(y_test, pred, zero_division=0),
        "MCC":       matthews_corrcoef(y_test, pred),
    })

scoreboard = pd.DataFrame(rows)

try:                      # colour-graded table when the pandas Styler is available
    display(scoreboard.round(4).style
            .background_gradient(cmap="Greens", subset=METRICS)
            .format({m: "{:.4f}" for m in METRICS}))
except (ImportError, AttributeError):
    print(scoreboard.round(4).to_string(index=False))

### 5.1 Why MCC is the tie-breaker here

At 37% positives the classes are only mildly imbalanced, but accuracy still
flatters weak models: a classifier that called everything benign would score
63% accuracy and an MCC of 0. MCC uses all four confusion-matrix cells, so it
is the honest single number for ranking these five.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(19, 3.4))
for ax, (name, model) in zip(axes, fitted.items()):
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt="d", cbar=False, cmap="Blues", ax=ax,
                xticklabels=["Ben","Mal"], yticklabels=["Ben","Mal"])
    ax.set_title(f"{name}\nFN={cm[1,0]}", fontsize=9)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
plt.tight_layout(); plt.show()

In [ ]:
plt.figure(figsize=(6.4, 5))
for name, model in fitted.items():
    proba = model.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, lw=1.9,
             label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")
plt.plot([0,1],[0,1],"k--",lw=.9,label="Chance")
plt.xlabel("False positive rate"); plt.ylabel("True positive rate")
plt.title("ROC curves — held-out screening set")
plt.legend(fontsize=8, loc="lower right"); plt.tight_layout(); plt.show()

In [ ]:
melted = scoreboard.melt(id_vars="ML Model Name", var_name="Metric", value_name="Score")
plt.figure(figsize=(11, 4))
sns.barplot(melted, x="Metric", y="Score", hue="ML Model Name")
plt.ylim(0.75, 1.01); plt.title("Metric comparison across the five models")
plt.legend(fontsize=7, ncol=3, loc="lower left"); plt.tight_layout(); plt.show()

### 5.2 Detailed report for the winning model

In [ ]:
winner = scoreboard.sort_values(["MCC","AUC"], ascending=False).iloc[0]["ML Model Name"]
print(f"Winner by MCC then AUC: {winner}\n")
print(classification_report(y_test, fitted[winner].predict(X_test),
                            target_names=["Benign","Malignant"], digits=4))

### 5.3 What logistic regression actually learned

With an L1 penalty the model zeroes out most of the 30 redundant columns and
keeps a compact, readable set of drivers — which is the practical reason it
beats the forest here rather than an accident of the split.

In [ ]:
lr = fitted["Logistic Regression"]
coefs = pd.Series(lr.named_steps["clf"].coef_[0], index=X.columns)
kept = coefs[coefs != 0].sort_values(key=abs, ascending=False)
print(f"{(coefs == 0).sum()} of {len(coefs)} coefficients driven to exactly zero\n")

plt.figure(figsize=(7, 5))
kept.head(14).sort_values().plot(kind="barh", color="#4c72b0")
plt.title("Largest surviving L1 coefficients (log-odds of malignancy)")
plt.tight_layout(); plt.show()

In [ ]:
rf = fitted["Random Forest (Ensemble)"]
imp = pd.Series(rf.named_steps["clf"].feature_importances_,
                index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(7, 5))
imp.head(14).sort_values().plot(kind="barh", color="#55a868")
plt.title("Random Forest — top 14 feature importances")
plt.tight_layout(); plt.show()

## 6 · Persist artefacts

Saves the fitted pipelines, the feature order (used by the app to validate an
upload) and the held-out screening CSV consumed by the Streamlit front end.

In [ ]:
ART = Path("artifacts"); ART.mkdir(exist_ok=True)
for name, model in fitted.items():
    slug = name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    joblib.dump(model, ART / f"{slug}.joblib")
joblib.dump(list(X.columns), ART / "feature_order.joblib")
scoreboard.to_csv(ART / "metrics_summary.csv", index=False)

test_frame = X_test.copy(); test_frame[TARGET] = y_test.values
test_frame.to_csv("../test_data.csv", index=False)
print("artefacts saved:", sorted(p.name for p in ART.iterdir()))

## 7 · Conclusions

| Model | Observation |
|---|---|
| Logistic Regression | Best overall — MCC 0.9700, AUC 0.9979, only 1 missed malignancy. The classes are close to linearly separable once standardised, and the L1 penalty prunes the redundant geometry columns. |
| Decision Tree | Weakest — MCC 0.8193, 9 false negatives. A single axis-aligned tree must commit to hard thresholds on correlated features, and pruning to depth 5 to stop overfitting costs recall. |
| kNN | Strong runner-up — MCC 0.9555 and perfect precision (0 false positives), but 3 malignancies missed. Distance-weighted voting over 9 neighbours works well after scaling. |
| Naive Bayes | AUC 0.9885 but MCC only 0.8343 — it ranks patients well yet its default 0.5 cut-off is poorly placed, because the conditional-independence assumption is badly violated by the mean/SE/worst triplets. |
| Random Forest | MCC 0.9098, 4 false negatives. Robust and needs no scaling, but on 426 training rows with near-linear structure the extra variance reduction does not beat a well-regularised linear model. |

**Overall winner: Logistic Regression** — highest score on all six metrics
except precision (where kNN is perfect), the single lowest false-negative count,
and the cheapest, most interpretable model of the five.